<img src='https://hammondm.github.io/hltlogo1.png' style="float:right">

LING 593A-011<br>
Fall 2025<br>
Davo Acevedo-Cardona

# Green Thumbs-Up Superset (PACs)

This notebook is a test sample to upload all the CSV to firebase.

Sai, Lindsey and I still need to agree where the data will be uploaded (Firebase project).


# Imports

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata

In [2]:
ROOT = Path(r"C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 9")

INPUTS  = ROOT / "inputs"
OUTPUTS = ROOT / "outputs"
LOGS    = ROOT / "logs"

INPUTS.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(parents=True, exist_ok=True)

exec_path  = INPUTS / "executive_final.csv"
xwalk_path = INPUTS / "company_ticker_crosswalk.csv"

print("ROOT:", ROOT)
print("INPUTS:", INPUTS)
print("OUTPUTS:", OUTPUTS)
print("LOGS:", LOGS)

ROOT: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 9
INPUTS: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 9\inputs
OUTPUTS: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 9\outputs
LOGS: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 9\logs


# Load Inputs

In [3]:
exec_df = pd.read_csv(exec_path)
xwalk   = pd.read_csv(xwalk_path)

print("executives rows:", len(exec_df), "cols:", exec_df.shape[1])
print("xwalk rows:", len(xwalk), "cols:", xwalk.shape[1])

display(exec_df.head(3))
display(xwalk.head(3))

executives rows: 11283 cols: 6
xwalk rows: 21814 cols: 2


,company,filing_type,executive_name,executive_title,confidence,executive_title_clean
0,1 800 FLOWERS COM INC,8-K,Moose Munch,Coo,spacy,Chief Operating Officer
1,"10x Genomics, Inc.",8-K,James Wilbur,Director,spacy,Director
2,1606 CORP.,8-K,Gregory Lambrecht,Chief Executive Officer,low,Chief Executive Officer


,TICKER,COMPANY_NAME
0,A,Agilent Technologies Inc
1,AA,Alcoa Corp
2,AAAIF,Alternative Investment Trust


# Merge

In [4]:
exec_df["company_clean_basic"] = (
    exec_df["company"].astype(str).str.upper().str.strip()
)

xwalk["company_clean_basic"] = (
    xwalk["COMPANY_NAME"].astype(str).str.upper().str.strip()
)

baseline = exec_df.merge(
    xwalk[["company_clean_basic", "TICKER"]],
    on="company_clean_basic",
    how="left"
)

baseline_missing_rate = baseline["TICKER"].isna().mean()
print("Baseline missing ticker rate:", baseline_missing_rate)

Baseline missing ticker rate: 0.6490055294214739


# Company normalization function

In [5]:
def norm_company(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).upper().strip()
    s = unicodedata.normalize("NFKD", s)
    s = s.replace("&", " AND ")

    # Remove SEC-style jurisdiction tags: "/DE/", "/MD/", etc.
    s = re.sub(r"/[A-Z]{2,3}/", " ", s)

    # Remove punctuation -> spaces
    s = re.sub(r"[^A-Z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    # Strip common suffixes iteratively
    suffixes = [
        " INCORPORATED"," INC",
        " CORPORATION"," CORP",
        " COMPANY"," CO",
        " LIMITED"," LTD",
        " L L C"," LLC",
        " L P"," LP",
        " P L C"," PLC",
        " HOLDINGS"," HOLDING",
        " GROUP",
        " THE",
    ]

    changed = True
    while changed:
        changed = False
        for suf in suffixes:
            if s.endswith(suf):
                s = s[: -len(suf)].strip()
                changed = True

    return s

# Normalize both datasets + build safe ticker map

In [6]:
# Normalize company names
exec_df["company_norm"] = exec_df["company"].map(norm_company)
xwalk["company_norm"]   = xwalk["COMPANY_NAME"].map(norm_company)

# Build a mapping that preserves multiple tickers (prevents row explosion)
ticker_map = (
    xwalk.groupby("company_norm")["TICKER"]
         .apply(lambda s: sorted(set(s.dropna().astype(str))))
         .reset_index()
         .rename(columns={"TICKER": "ticker_candidates"})
)

exec_with_ticker = exec_df.merge(ticker_map, on="company_norm", how="left")

# Deterministic primary ticker choice + ambiguity flag
exec_with_ticker["TICKER"] = exec_with_ticker["ticker_candidates"].apply(
    lambda lst: lst[0] if isinstance(lst, list) and len(lst) else pd.NA
)
exec_with_ticker["ticker_ambiguous"] = exec_with_ticker["ticker_candidates"].apply(
    lambda lst: isinstance(lst, list) and len(lst) > 1
)

missing_rate = exec_with_ticker["TICKER"].isna().mean()
ambig_rate   = exec_with_ticker["ticker_ambiguous"].mean()

print("After normalization missing ticker rate:", missing_rate)
print("Ambiguous ticker rate:", ambig_rate)
display(exec_with_ticker.head(5))

After normalization missing ticker rate: 0.13046175662501108
Ambiguous ticker rate: 0.08933794203669237


,company,filing_type,executive_name,executive_title,confidence,executive_title_clean,company_clean_basic,company_norm,ticker_candidates,TICKER,ticker_ambiguous
0,1 800 FLOWERS COM INC,8-K,Moose Munch,Coo,spacy,Chief Operating Officer,1 800 FLOWERS COM INC,1 800 FLOWERS COM,[FLWS],FLWS,False
1,"10x Genomics, Inc.",8-K,James Wilbur,Director,spacy,Director,"10X GENOMICS, INC.",10X GENOMICS,[TXG],TXG,False
2,1606 CORP.,8-K,Gregory Lambrecht,Chief Executive Officer,low,Chief Executive Officer,1606 CORP.,1606,[CBDW],CBDW,False
3,"1895 Bancorp of Wisconsin, Inc. /MD/",8-K,David Ball,Chief Executive Officer,spacy,Chief Executive Officer,"1895 BANCORP OF WISCONSIN, INC. /MD/",1895 BANCORP OF WISCONSIN,[BCOW],BCOW,False
4,"1stdibs.com, Inc.",8-K,Everette Taylor,Director,spacy,Director,"1STDIBS.COM, INC.",1STDIBS COM,[DIBS],DIBS,False


# QA exports

In [8]:
# Unmatched is fine (no list column involved)
unmatched_companies = (
    exec_with_ticker.loc[exec_with_ticker["TICKER"].isna(), ["company", "company_norm"]]
        .drop_duplicates()
        .sort_values(["company_norm", "company"])
)

# For ambiguous, convert list -> string so pandas can dedupe
ambiguous_tmp = exec_with_ticker.loc[
    exec_with_ticker["ticker_ambiguous"],
    ["company", "company_norm", "ticker_candidates"]
].copy()

ambiguous_tmp["ticker_candidates_str"] = ambiguous_tmp["ticker_candidates"].apply(
    lambda x: "|".join(map(str, x)) if isinstance(x, list) else str(x)
)

ambiguous_companies = (
    ambiguous_tmp[["company", "company_norm", "ticker_candidates_str"]]
        .drop_duplicates()
        .sort_values(["company_norm", "company"])
)

unmatched_path = LOGS / "task9_unmatched_companies_after_norm.csv"
ambig_path     = LOGS / "task9_ambiguous_company_to_ticker.csv"

unmatched_companies.to_csv(unmatched_path, index=False)
ambiguous_companies.to_csv(ambig_path, index=False)

print("Wrote:", unmatched_path)
print("Wrote:", ambig_path)

print("\nTop 20 unmatched:")
display(unmatched_companies.head(20))

print("\nTop 20 ambiguous:")
display(ambiguous_companies.head(20))

Wrote: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 9\logs\task9_unmatched_companies_after_norm.csv
Wrote: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 9\logs\task9_ambiguous_company_to_ticker.csv

Top 20 unmatched:


,company,company_norm
6,21Shares Core Ethereum ETF,21SHARES CORE ETHEREUM ETF
10,"2seventy bio, Inc.",2SEVENTY BIO
46,A.K.A. BRANDS HOLDING CORP.,A K A BRANDS
53,"Aadi Bioscience, Inc.",AADI BIOSCIENCE
59,"Abacus Life, Inc.",ABACUS LIFE
74,abrdn Gold ETF Trust,ABRDN GOLD ETF TRUST
86,"Accelerate Diagnostics, Inc",ACCELERATE DIAGNOSTICS
91,"Accolade, Inc.",ACCOLADE
98,"ACELYRIN, Inc.",ACELYRIN
101,Aceztech Corp,ACEZTECH



Top 20 ambiguous:


,company,company_norm,ticker_candidates_str
38,"5E Advanced Materials, Inc.",5E ADVANCED MATERIALS,FEAM|FEAV
115,ACRES Commercial Realty Corp.,ACRES COMMERCIAL REALTY,ACR|ACR.PRC|ACR.PRD
258,"AFFILIATED MANAGERS GROUP, INC.",AFFILIATED MANAGERS,AMG|MGR|MGRB|MGRD|MGRE
266,"AG Mortgage Investment Trust, Inc.",AG MORTGAGE INVESTMENT TRUST,MITN|MITP|MITT|MITT.PRA|MITT.PRB|MITT.PRC
287,AGREE REALTY CORP,AGREE REALTY,ADC|ADC.PRA
288,Agriculture & Natural Solutions Acquisition Corp,AGRICULTURE AND NATURAL SOLUTIONS ACQUISITION,ANSC|ANSCU
315,"Aimei Health Technology Co., Ltd.",AIMEI HEALTH TECHNOLOGY,AFJK|AFJKR|AFJKU
353,ALBEMARLE CORP,ALBEMARLE,ALB|ALB.PRA
356,Alchemy Investments Acquisition Corp 1,ALCHEMY INVESTMENTS ACQUISITION CORP 1,ALCY|ALCYU
428,ALLSTATE CORP,ALLSTATE,ALL|ALL.PRB|ALL.PRH|ALL.PRI|ALL.PRJ


In [9]:
# Unmatched is fine (no list column involved)
unmatched_companies = (
    exec_with_ticker.loc[exec_with_ticker["TICKER"].isna(), ["company", "company_norm"]]
        .drop_duplicates()
        .sort_values(["company_norm", "company"])
)

# For ambiguous, convert list -> string so pandas can dedupe
ambiguous_tmp = exec_with_ticker.loc[
    exec_with_ticker["ticker_ambiguous"],
    ["company", "company_norm", "ticker_candidates"]
].copy()

ambiguous_tmp["ticker_candidates_str"] = ambiguous_tmp["ticker_candidates"].apply(
    lambda x: "|".join(map(str, x)) if isinstance(x, list) else str(x)
)

ambiguous_companies = (
    ambiguous_tmp[["company", "company_norm", "ticker_candidates_str"]]
        .drop_duplicates()
        .sort_values(["company_norm", "company"])
)

unmatched_path = LOGS / "task9_unmatched_companies_after_norm.csv"
ambig_path     = LOGS / "task9_ambiguous_company_to_ticker.csv"

unmatched_companies.to_csv(unmatched_path, index=False)
ambiguous_companies.to_csv(ambig_path, index=False)

print("Wrote:", unmatched_path)
print("Wrote:", ambig_path)

print("\nTop 20 unmatched:")
display(unmatched_companies.head(20))

print("\nTop 20 ambiguous:")
display(ambiguous_companies.head(20))

Wrote: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 9\logs\task9_unmatched_companies_after_norm.csv
Wrote: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 9\logs\task9_ambiguous_company_to_ticker.csv

Top 20 unmatched:


,company,company_norm
6,21Shares Core Ethereum ETF,21SHARES CORE ETHEREUM ETF
10,"2seventy bio, Inc.",2SEVENTY BIO
46,A.K.A. BRANDS HOLDING CORP.,A K A BRANDS
53,"Aadi Bioscience, Inc.",AADI BIOSCIENCE
59,"Abacus Life, Inc.",ABACUS LIFE
74,abrdn Gold ETF Trust,ABRDN GOLD ETF TRUST
86,"Accelerate Diagnostics, Inc",ACCELERATE DIAGNOSTICS
91,"Accolade, Inc.",ACCOLADE
98,"ACELYRIN, Inc.",ACELYRIN
101,Aceztech Corp,ACEZTECH



Top 20 ambiguous:


,company,company_norm,ticker_candidates_str
38,"5E Advanced Materials, Inc.",5E ADVANCED MATERIALS,FEAM|FEAV
115,ACRES Commercial Realty Corp.,ACRES COMMERCIAL REALTY,ACR|ACR.PRC|ACR.PRD
258,"AFFILIATED MANAGERS GROUP, INC.",AFFILIATED MANAGERS,AMG|MGR|MGRB|MGRD|MGRE
266,"AG Mortgage Investment Trust, Inc.",AG MORTGAGE INVESTMENT TRUST,MITN|MITP|MITT|MITT.PRA|MITT.PRB|MITT.PRC
287,AGREE REALTY CORP,AGREE REALTY,ADC|ADC.PRA
288,Agriculture & Natural Solutions Acquisition Corp,AGRICULTURE AND NATURAL SOLUTIONS ACQUISITION,ANSC|ANSCU
315,"Aimei Health Technology Co., Ltd.",AIMEI HEALTH TECHNOLOGY,AFJK|AFJKR|AFJKU
353,ALBEMARLE CORP,ALBEMARLE,ALB|ALB.PRA
356,Alchemy Investments Acquisition Corp 1,ALCHEMY INVESTMENTS ACQUISITION CORP 1,ALCY|ALCYU
428,ALLSTATE CORP,ALLSTATE,ALL|ALL.PRB|ALL.PRH|ALL.PRI|ALL.PRJ


In [13]:
def choose_common_ticker(cands):
    """
    cands: list[str] or NaN
    Returns a single 'best' ticker for company-level work.
    """
    if not isinstance(cands, list) or len(cands) == 0:
        return pd.NA

    # Normalize to uppercase strings
    cands = [str(x).strip().upper() for x in cands if pd.notna(x)]

    def is_common(t):
        # Exclude preferred shares (.PRB, -PRA, etc.)
        if re.search(r"(\.PR|\.PRA|\.PRB|\.PRC|\.PRD|\.PRE|\.PRF|\.PRG|\.PRH|\.PRI|\.PRJ|\.PRK|\.PRL)\b", t):
            return False
        if re.search(r"(\-PR[A-Z]?)\b", t):
            return False
        # Exclude SPAC-like suffixes
        if re.search(r"\b(W|WS|WT|R|RT|U)\b$", t):
            return False
        return True

    commons = [t for t in cands if is_common(t)]

    if len(commons) > 0:
        return sorted(commons)[0]  # deterministic choice

    return sorted(cands)[0]  # fallback

In [14]:
df = exec_with_ticker.copy()

df["TICKER_COMMON"] = df["ticker_candidates"].apply(choose_common_ticker)

print("TICKER missing rate:", df["TICKER"].isna().mean())
print("TICKER_COMMON missing rate:", df["TICKER_COMMON"].isna().mean())

# How many ambiguous companies now have a usable common ticker?
ambig_has_common = df.loc[df["ticker_ambiguous"], "TICKER_COMMON"].notna().mean()
print("Among ambiguous, % with TICKER_COMMON:", ambig_has_common)

TICKER missing rate: 0.13046175662501108
TICKER_COMMON missing rate: 0.13046175662501108
Among ambiguous, % with TICKER_COMMON: 1.0


In [15]:
ambig_out = (
    df.loc[df["ticker_ambiguous"], ["company", "company_norm", "ticker_candidates"]]
      .drop_duplicates(subset=["company_norm"])
      .copy()
)

ambig_out["ticker_candidates_str"] = ambig_out["ticker_candidates"].apply(
    lambda x: "|".join(map(str, x)) if isinstance(x, list) else str(x)
)

# Recompute chosen common ticker at the company_norm level
ambig_out["TICKER_COMMON"] = ambig_out["ticker_candidates"].apply(choose_common_ticker)

ambig_out = ambig_out[["company", "company_norm", "ticker_candidates_str", "TICKER_COMMON"]].sort_values(
    ["company_norm", "company"]
)

ambig_common_path = LOGS / "task9_ambiguous_company_to_ticker_with_common.csv"
ambig_out.to_csv(ambig_common_path, index=False)
print("Wrote:", ambig_common_path)

Wrote: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 9\logs\task9_ambiguous_company_to_ticker_with_common.csv


In [16]:
unmatched = pd.read_csv(LOGS / "task9_unmatched_companies_after_norm.csv")

pattern = r"\b(ETF|TRUST|FUND|PORTFOLIO|ETN)\b"
unmatched["looks_like_fund_or_etf"] = unmatched["company_norm"].str.contains(pattern, regex=True, na=False)

print("Unmatched total:", len(unmatched))
print("Likely ETF/fund/trust:", unmatched["looks_like_fund_or_etf"].mean())

unmatched_triage_path = LOGS / "task9_unmatched_companies_triage.csv"
unmatched.to_csv(unmatched_triage_path, index=False)
print("Wrote:", unmatched_triage_path)

display(unmatched.head(20))

Unmatched total: 573
Likely ETF/fund/trust: 0.06282722513089005
Wrote: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 9\logs\task9_unmatched_companies_triage.csv


C:\Users\ateni\AppData\Local\Temp\ipykernel_26020\3704365649.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  unmatched["looks_like_fund_or_etf"] = unmatched["company_norm"].str.contains(pattern, regex=True, na=False)


,company,company_norm,looks_like_fund_or_etf
0,21Shares Core Ethereum ETF,21SHARES CORE ETHEREUM ETF,True
1,"2seventy bio, Inc.",2SEVENTY BIO,False
2,A.K.A. BRANDS HOLDING CORP.,A K A BRANDS,False
3,"Aadi Bioscience, Inc.",AADI BIOSCIENCE,False
4,"Abacus Life, Inc.",ABACUS LIFE,False
5,abrdn Gold ETF Trust,ABRDN GOLD ETF TRUST,True
6,"Accelerate Diagnostics, Inc",ACCELERATE DIAGNOSTICS,False
7,"Accolade, Inc.",ACCOLADE,False
8,"ACELYRIN, Inc.",ACELYRIN,False
9,Aceztech Corp,ACEZTECH,False


# Merge Execs

In [17]:
out_exec_path = OUTPUTS / "task9_executives_with_ticker_enriched.csv"
df.to_csv(out_exec_path, index=False)
print("Saved:", out_exec_path)

Saved: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 9\outputs\task9_executives_with_ticker_enriched.csv


# Prep executive name fields for donation matching

In [18]:
def norm_person_name(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = unicodedata.normalize("NFKD", s)
    s = re.sub(r"\s+", " ", s)
    return s

df = exec_with_ticker.copy()

df["exec_name_raw"] = df["executive_name"].map(norm_person_name)
df["exec_name_up"]  = df["exec_name_raw"].str.upper()

# Remove commas and common suffixes (very conservative)
df["exec_name_up"] = (
    df["exec_name_up"]
      .str.replace(",", " ", regex=False)
      .str.replace(r"\b(JR|SR|II|III|IV|V)\b\.?", "", regex=True)
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

# Split first token as first_name, last token as last_name
parts = df["exec_name_up"].str.split(" ")

df["exec_first_name"] = parts.str[0]
df["exec_last_name"]  = parts.str[-1]

display(df[["executive_name", "exec_first_name", "exec_last_name"]].head(10))

,executive_name,exec_first_name,exec_last_name
0,Moose Munch,MOOSE,MUNCH
1,James Wilbur,JAMES,WILBUR
2,Gregory Lambrecht,GREGORY,LAMBRECHT
3,David Ball,DAVID,BALL
4,Everette Taylor,EVERETTE,TAYLOR
5,David Rosenblatt,DAVID,ROSENBLATT
6,Ophelia Snyder,OPHELIA,SNYDER
7,Jody Mettler,JODY,METTLER
8,Rachel Anderika,RACHEL,ANDERIKA
9,Larry Firestone,LARRY,FIRESTONE


# Load donations extract when ready

In [19]:
don_path = INPUTS / "individual_donations_2020_2026.csv"

if don_path.exists():
    don = pd.read_csv(don_path)
    print("donations rows:", len(don), "cols:", don.shape[1])
    display(don.head(3))
else:
    print("Missing donations file:", don_path)

Missing donations file: C:\Users\ateni\Desktop\U of A\LING 593A Internship-Hum Lang Tech\Task 9\inputs\individual_donations_2020_2026.csv
